# Metrics for First N Samples

Re-compute evaluation metrics restricted to the **first N unique samples** (by `frame_index`)
from a benchmark run, without re-running the full benchmark.

**Why this is needed:** results are saved in two separate files — `evaluation_results.csv`
(correct predictions) and `evaluation_wrongs.csv` (wrong/error predictions) — with no
common "label" to slice by sample count. By sorting the union of `frame_index` values from
both files, we can reconstruct the original sample ordering and restrict analysis to the
first N.

In [ ]:
from pathlib import Path
import re

# ── Configuration ─────────────────────────────────────────────────────────────
# Number of samples to consider (set to None to use all samples)
N = 150

# Point this to ANY timestamped result folder that contains the two CSV files.
# Example layout: results/AtomWorld/<model>/<action>/<timestamp>/
# RESULTS_DIR = Path(
#     r"d:\Codes\AtomWorld\results\AtomWorld\deepseek_chat\add_atom_action\20260325_123216"
# )

DIR = Path(r"d:\Codes\AtomWorld\results\AtomWorld\deepseek_chat\move_towards_atom_action")
pattern = re.compile(r"20260325_(\d{6})$")

best = None
best_ts = -1
for p in DIR.iterdir():
    m = pattern.search(p.name)
    if m:
        ts = int(m.group(1))
        if ts > best_ts:
            best_ts = ts
            best = p

RESULTS_DIR = best

# ─────────────────────────────────────────────────────────────────────────────

In [90]:
import pandas as pd

# ── Load raw data ─────────────────────────────────────────────────────────────
df_results = pd.read_csv(RESULTS_DIR / "evaluation_results.csv")
df_wrongs  = pd.read_csv(RESULTS_DIR / "evaluation_wrongs.csv")

# ── Determine the first N unique frame indices ────────────────────────────────
# Each sample appears in exactly one of the two files; frame_index identifies it.
# Sorting the union of both files' frame_index values reconstructs the run order.
all_frame_indices = sorted(
    set(df_results["frame_index"]).union(set(df_wrongs["frame_index"]))
)
total_available = len(all_frame_indices)

if N is None or N >= total_available:
    selected_frames = set(all_frame_indices)
    n_used = total_available
    print(f"Using all {total_available} available samples.")
else:
    selected_frames = set(all_frame_indices[:N])
    n_used = N
    print(f"Using first {N} of {total_available} available samples.")

# ── Filter each file to selected frames ──────────────────────────────────────
df_res_N   = df_results[df_results["frame_index"].isin(selected_frames)].copy()
df_wrong_N = df_wrongs[df_wrongs["frame_index"].isin(selected_frames)].copy()

print(f"  Correct results : {len(df_res_N)}")
print(f"  Wrong results   : {len(df_wrong_N)}")
print(f"  Sum check (== {n_used}): {len(df_res_N) + len(df_wrong_N)}")

Using all 150 available samples.
  Correct results : 97
  Wrong results   : 53
  Sum check (== 150): 150


In [91]:
import numpy as np

# ── Core metrics ──────────────────────────────────────────────────────────────
total       = len(df_res_N) + len(df_wrong_N)
n_correct   = len(df_res_N)
n_wrong     = len(df_wrong_N)
accuracy    = n_correct / total if total else float("nan")
error_rate  = n_wrong   / total if total else float("nan")

# RMSD and max_diff stats (only from correct predictions)
rmsd_stats     = df_res_N["rmsd"].describe()    if len(df_res_N) else pd.Series(dtype=float)
max_diff_stats = df_res_N["max_diff"].describe() if len(df_res_N) else pd.Series(dtype=float)

# Wrong-type breakdown
wrong_type_counts = (
    df_wrong_N["wrong_type"].value_counts()
    if "wrong_type" in df_wrong_N.columns and len(df_wrong_N)
    else pd.Series(dtype=int)
)

# ── Summary printout ──────────────────────────────────────────────────────────
sep = "─" * 50
print(sep)
print(f"  Samples analysed  : {total} (first {n_used} unique frame indices)")
print(sep)
print(f"  Correct           : {n_correct}  ({accuracy:.1%})")
print(f"  Wrong / error     : {n_wrong}   ({error_rate:.1%})")
# print(f"  Strict accuracy   : {n_strict}  ({strict_accuracy:.1%})  [max_diff < {MAX_DIFF_THRESHOLD}]")
print(sep)

if len(df_res_N):
    print("  RMSD (correct predictions):")
    print(f"    mean={rmsd_stats['mean']:.6f}  std={rmsd_stats['std']:.6f}"
          f"  min={rmsd_stats['min']:.6f}  max={rmsd_stats['max']:.6f}")
    print("  max_diff (correct predictions):")
    print(f"    mean={max_diff_stats['mean']:.6f}  std={max_diff_stats['std']:.6f}"
          f"  min={max_diff_stats['min']:.6f}  max={max_diff_stats['max']:.6f}")
    print(sep)

if len(wrong_type_counts):
    print("  Wrong-type breakdown:")
    for wtype, cnt in wrong_type_counts.items():
        print(f"    {wtype:<30} {cnt:>4}  ({cnt/total:.1%})")
    print(sep)

──────────────────────────────────────────────────
  Samples analysed  : 150 (first 150 unique frame indices)
──────────────────────────────────────────────────
  Correct           : 97  (64.7%)
  Wrong / error     : 53   (35.3%)
──────────────────────────────────────────────────
  RMSD (correct predictions):
    mean=0.026017  std=0.054261  min=0.000000  max=0.253930
  max_diff (correct predictions):
    mean=0.096469  std=0.186299  min=0.000000  max=0.930130
──────────────────────────────────────────────────
  Wrong-type breakdown:
    StructureMismatch                50  (33.3%)
    AtomCountMismatch                 3  (2.0%)
──────────────────────────────────────────────────
